# **pix2pix: caras a partir de máscaras de segmentación**

## **0. Objetivo del notebook**

Este notebook entrena desde cero un modelo **pix2pix** (Isola et al., 2017) que genera caras en escala de grises de `64x64` píxeles a partir de las **máscaras de segmentación** de CelebAMask-HQ. Se parte de la arquitectura de `AE_complex2`: su encoder y su decoder convolucionales se convierten en el generador (una U-Net), se añade un discriminador PatchGAN y el entrenamiento pasa a ser adversarial.

### **Qué cambia respecto a AE 2**

| | AE 2 (`AE_complex2`) | pix2pix (este notebook) |
|---|---|---|
| Tarea | reconstruir la cara de entrada | traducir máscara → cara |
| Entrada | cara `64x64x1` | máscara `64x64x8` (un canal por clase) |
| Cuello de botella | vector latente `Dense` de 256 | mapa convolucional `4x4x512` |
| Skip connections | no | sí (U-Net) |
| Aleatoriedad al generar | muestreo del latente (kNN, GMM...) | dropout activo también en inferencia |
| Salida | `sigmoid` en `[0, 1]` | `tanh` en `[-1, 1]` |
| Pérdida | L1 + SSIM | adversarial (PatchGAN) + λ·(L1 + SSIM) |
| Selección del modelo | `val_loss` + `EarlyStopping` | FID en validación cada `FID_EVERY` épocas |
| Evaluación | FID | FID, KID, L1/SSIM/PSNR frente a la cara real, diversidad y originalidad |

---

## **1. Configuración e imports**

Parámetros principales:

- `CLASSES`: las 8 clases de máscara del dataset. Cada una ocupa un canal de la entrada del generador.
- `LAMBDA_REC`, `LEARNING_RATE` y `BETA_1`: peso de la pérdida de reconstrucción frente a la adversarial y configuración de Adam, con los valores del artículo de pix2pix.
- `N_DROPOUT_BLOCKS` y `DROPOUT_RATE`: cuántos bloques del decoder llevan dropout (la fuente de aleatoriedad del generador).
- `FID_EVERY`: cada cuántas épocas se calcula el FID en validación para guardar el mejor generador.
- `N_EVAL_REPEATS`: el generador es estocástico, así que las métricas finales se calculan varias veces con semillas distintas y se reporta media y desviación.

In [ ]:
%pip install -q "torchmetrics[image]"
%pip install -q opencv-python

In [ ]:
import os
import sys
import json
import time
import zipfile

import cv2
import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
from matplotlib.patches import Patch

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

import torch
from torchmetrics.image.fid import FrechetInceptionDistance
from torchmetrics.image.kid import KernelInceptionDistance

# TensorFlow reserva por defecto toda la memoria de la GPU; así deja sitio a PyTorch (FID/KID)
for gpu in tf.config.list_physical_devices("GPU"):
    tf.config.experimental.set_memory_growth(gpu, True)

print("GPUs TensorFlow:", tf.config.list_physical_devices("GPU"))
print("CUDA PyTorch:   ", torch.cuda.is_available())

In [ ]:
# Configuración común
SEED = 1234
IMG_SIZE = 64
TOTAL_IMAGES = 30000              # imágenes 0.jpg ... 29999.jpg con sus máscaras 00000_*.png ... 29999_*.png
VAL_FRACTION = 0.05               # validación: selección del mejor generador (FID)
TEST_FRACTION = 0.05              # test: métricas finales, no se usa para ninguna decisión

CLASSES = ("hair", "hat", "l_ear", "r_ear", "l_eye", "r_eye", "nose", "mouth")
N_CLASSES = len(CLASSES)

# Específico
NOTEBOOK_NAME = "pix2pix"
TRAIN_MODEL = True                # False: carga los modelos guardados y no entrena

EPOCHS = 40
BATCH_SIZE = 64
LEARNING_RATE = 2e-4
BETA_1 = 0.5
LAMBDA_REC = 100.0
N_DROPOUT_BLOCKS = 2
DROPOUT_RATE = 0.5

# Evaluación
FID_EVERY = 2                     # épocas entre cálculos de FID en validación
FID_BATCH_SIZE = 64
KID_SUBSET_SIZE = 100
N_EVAL_REPEATS = 3
N_DIVERSITY = 8                   # muestras por máscara en el estudio de diversidad
N_DIVERSITY_MASKS = 300
N_ORIGINALITY = 500

# Rutas: en Colab todo se guarda en Drive para no perderlo al cerrar la sesión
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/Practica2"
    DATASET_ZIP = f"{PROJECT_DIR}/CelebAMask.zip"     # zip con las carpetas images/ y masks/
    DATASET_DIR = "/content/dataset"                   # se descomprime en el disco local de Colab
else:
    PROJECT_DIR = "."
    DATASET_ZIP = None
    DATASET_DIR = "mg-practica-1-generando-caras/CelebAMask"

CACHE_PATH = f"{PROJECT_DIR}/cache/celebamask_{IMG_SIZE}_gray.npz"
CHECKPOINT_DIR = f"{PROJECT_DIR}/checkpoints"
RESULTS_DIR = f"{PROJECT_DIR}/results"
METRICS_PATH = f"{RESULTS_DIR}/{NOTEBOOK_NAME}_metrics.json"
GENERATOR_BEST_PATH = f"{CHECKPOINT_DIR}/{NOTEBOOK_NAME}_generator_best.keras"
GENERATOR_LAST_PATH = f"{CHECKPOINT_DIR}/{NOTEBOOK_NAME}_generator_last.keras"
DISCRIMINATOR_LAST_PATH = f"{CHECKPOINT_DIR}/{NOTEBOOK_NAME}_discriminator_last.keras"
HISTORY_PATH = f"{CHECKPOINT_DIR}/{NOTEBOOK_NAME}_history.json"


def save_fig(name):
    os.makedirs(RESULTS_DIR, exist_ok=True)
    plt.savefig(f"{RESULTS_DIR}/{name}.png", dpi=150, bbox_inches="tight")


keras.utils.set_random_seed(SEED)

---

## **2. Carga de datos y split**

Cada ejemplo del dataset está formado por:

- Una **cara** `images/{i}.jpg` de `64x64` en color, que se carga en escala de grises.
- Un fichero de **máscara por cada clase presente**, `masks/{i:05d}_{clase}.png`. Si a una cara le falta una clase (por ejemplo, no se le ven las orejas), ese fichero no existe y su canal queda a cero.

Las máscaras no son binarias: al reducirlas a `64x64` se suavizaron y sus bordes tienen valores intermedios entre 0 y 255. Por eso **no se convierten a una única etiqueta por píxel** (argmax): se apilan como 8 canales con valores en `[0, 1]`, lo que conserva los bordes suaves y los píxeles en los que dos clases se solapan.

Leer los ~167.000 PNG es lento (sobre todo desde Drive), así que la primera ejecución guarda una caché `.npz` y las siguientes la cargan directamente.

In [ ]:
def find_dataset_dir(root):
    for dirpath, dirnames, _ in os.walk(root):
        if "images" in dirnames and "masks" in dirnames:
            return dirpath
    raise FileNotFoundError(f"No se encuentran las carpetas images/ y masks/ dentro de {root}")


def load_from_disk(dataset_dir):
    images = np.empty((TOTAL_IMAGES, IMG_SIZE, IMG_SIZE), dtype=np.uint8)
    masks = np.zeros((TOTAL_IMAGES, IMG_SIZE, IMG_SIZE, N_CLASSES), dtype=np.uint8)
    mask_dir = os.path.join(dataset_dir, "masks")
    mask_files = set(os.listdir(mask_dir))

    for i in range(TOTAL_IMAGES):
        img_path = os.path.join(dataset_dir, "images", f"{i}.jpg")
        img = cv2.imread(img_path, cv2.IMREAD_GRAYSCALE)
        if img is None:
            raise FileNotFoundError(f"Error para cargar la imagen: {img_path}")
        if img.shape != (IMG_SIZE, IMG_SIZE):
            img = cv2.resize(img, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_AREA)
        images[i] = img

        for c, name in enumerate(CLASSES):
            filename = f"{i:05d}_{name}.png"
            if filename in mask_files:
                m = cv2.imread(os.path.join(mask_dir, filename), cv2.IMREAD_GRAYSCALE)
                if m.shape != (IMG_SIZE, IMG_SIZE):
                    # vecino más cercano: no mezcla valores de máscara al redimensionar
                    m = cv2.resize(m, (IMG_SIZE, IMG_SIZE), interpolation=cv2.INTER_NEAREST)
                masks[i, ..., c] = m
    return images, masks


t0 = time.time()
if os.path.exists(CACHE_PATH):
    with np.load(CACHE_PATH) as cache:
        images, masks = cache["images"], cache["masks"]
    origin = f"caché {CACHE_PATH}"
else:
    if IN_COLAB and not os.path.isdir(DATASET_DIR):
        with zipfile.ZipFile(DATASET_ZIP) as zf:
            zf.extractall(DATASET_DIR)
    images, masks = load_from_disk(find_dataset_dir(DATASET_DIR))
    os.makedirs(os.path.dirname(CACHE_PATH), exist_ok=True)
    np.savez_compressed(CACHE_PATH, images=images, masks=masks)
    origin = "disco (caché creada)"

print(f"Total cargado: {len(images)} pares, desde {origin} en {time.time() - t0:.1f}s")

In [ ]:
print(f"Forma de las caras:    {images.shape}")     # Esperado: (30000, 64, 64)
print(f"Forma de las máscaras: {masks.shape}")      # Esperado: (30000, 64, 64, 8)
print(f"Tipo de dato:          {images.dtype}, {masks.dtype}")   # Esperado: uint8, uint8
print(f"Rango de las caras:    {images.min()} - {images.max()}")  # Esperado: 0 - 255
print(f"Rango de las máscaras: {masks.min()} - {masks.max()}")    # Esperado: 0 - 255

In [ ]:
# Presencia y área media de cada clase
present = masks.max(axis=(1, 2)) > 127                     # (N, 8)
area = (masks > 127).mean(axis=(1, 2))                     # fracción de la imagen que ocupa cada clase
no_class = (masks.max(axis=-1) <= 127).mean()

df_classes = pd.DataFrame({
    "clase": CLASSES,
    "presente (%)": 100 * present.mean(axis=0),
    "área media si presente (%)": [100 * area[present[:, c], c].mean() for c in range(N_CLASSES)],
}).round(2)
print(df_classes.to_string(index=False))
print(f"\nPíxeles sin ninguna clase (piel, fondo, cuello, ropa...): {100 * no_class:.1f}%")

Las máscaras de este subconjunto **solo tienen 8 clases**: no hay piel, fondo, cejas, labios, cuello ni ropa. Más de la mitad de los píxeles no pertenece a ninguna clase, de modo que el generador tiene que deducir por sí mismo dónde termina la cara y empieza el fondo, a partir de la forma del pelo y de la posición de ojos, nariz y boca.

In [ ]:
# Paleta para dibujar las máscaras (izquierda y derecha con colores distintos)
PALETTE = np.array([
    [0.12, 0.12, 0.12],   # sin clase
    [0.55, 0.35, 0.15],   # hair
    [0.60, 0.20, 0.80],   # hat
    [1.00, 0.60, 0.00],   # l_ear
    [0.90, 0.85, 0.10],   # r_ear
    [0.10, 0.45, 1.00],   # l_eye
    [0.20, 0.90, 0.90],   # r_eye
    [0.20, 0.80, 0.20],   # nose
    [0.90, 0.10, 0.20],   # mouth
])


def mask_to_color(mask):
    mask = np.asarray(mask)
    if mask.dtype != np.uint8:
        mask = (np.clip(mask, 0.0, 1.0) * 255).astype(np.uint8)
    labels = np.where(mask.max(axis=-1) > 127, mask.argmax(axis=-1) + 1, 0)
    return PALETTE[labels]


def palette_legend(ax):
    handles = [Patch(color=PALETTE[c + 1], label=name) for c, name in enumerate(CLASSES)]
    ax.legend(handles=handles, loc="center left", bbox_to_anchor=(1.0, 0.5), fontsize=8)


rng = np.random.default_rng(42)
sample_indices = rng.choice(TOTAL_IMAGES, size=6, replace=False)
fig, axes = plt.subplots(3, 6, figsize=(12, 6.5))
for col, idx in enumerate(sample_indices):
    axes[0, col].imshow(images[idx], cmap="gray", vmin=0, vmax=255)
    axes[0, col].set_title(f"ID {idx}", fontsize=8)
    axes[1, col].imshow(mask_to_color(masks[idx]))
    axes[2, col].imshow(images[idx], cmap="gray", vmin=0, vmax=255)
    axes[2, col].imshow(mask_to_color(masks[idx]), alpha=0.45)
axes[0, 0].set_ylabel("Cara")
axes[1, 0].set_ylabel("Máscara")
axes[2, 0].set_ylabel("Superpuestas")
for ax in axes.flat:
    ax.set_xticks([])
    ax.set_yticks([])
palette_legend(axes[1, -1])
plt.tight_layout()
save_fig("dataset_ejemplos")
plt.show()

In [ ]:
# Split 90 / 5 / 5 con la misma semilla que en la práctica anterior
rng = np.random.default_rng(SEED)
indices = rng.permutation(TOTAL_IMAGES)
n_val = int(VAL_FRACTION * TOTAL_IMAGES)
n_test = int(TEST_FRACTION * TOTAL_IMAGES)
val_idx = indices[:n_val]
test_idx = indices[n_val:n_val + n_test]
train_idx = indices[n_val + n_test:]

img_train, mask_train = images[train_idx, ..., None], masks[train_idx]
img_val, mask_val = images[val_idx, ..., None], masks[val_idx]
img_test, mask_test = images[test_idx, ..., None], masks[test_idx]
del images, masks   # libera ~1 GB de RAM

print(f"Entrenamiento: {img_train.shape} | {mask_train.shape}")
print(f"Validación:    {img_val.shape} | {mask_val.shape}")
print(f"Test:          {img_test.shape} | {mask_test.shape}")

---

## **3. Pipeline de datos y aumento**

- Las caras pasan a `[-1, 1]`, el rango de la `tanh` de salida del generador. Las máscaras pasan a `[0, 1]`.
- El único aumento es el **volteo horizontal**, aplicado a la vez a la cara y a la máscara. Hay un detalle: `l_eye` y `l_ear` son el ojo y la oreja **izquierdos de la persona**, que en la imagen aparecen a la derecha (lo comprobamos abajo con la posición media de cada clase). Al voltear, el ojo izquierdo pasa a estar donde normalmente está el derecho, así que **también hay que intercambiar los canales** `l_eye`↔`r_eye` y `l_ear`↔`r_ear`. Si no se hace, la red recibe ejemplos contradictorios.

In [ ]:
def mean_column(mask_array, class_name):
    m = mask_array[..., CLASSES.index(class_name)] > 127
    cols = np.broadcast_to(np.arange(IMG_SIZE), m.shape)
    return cols[m].mean()


for name in ("l_eye", "r_eye", "l_ear", "r_ear"):
    print(f"{name}: columna media = {mean_column(mask_train[:2000], name):5.1f} (de 0 a {IMG_SIZE - 1})")

FLIP_PERM = [CLASSES.index(name) for name in
             ("hair", "hat", "r_ear", "l_ear", "r_eye", "l_eye", "nose", "mouth")]
print("\nPermutación de canales al voltear:", [CLASSES[i] for i in FLIP_PERM])

In [ ]:
def to_float(mask, image):
    mask = tf.cast(mask, tf.float32) / 255.0
    image = tf.cast(image, tf.float32) / 127.5 - 1.0
    return mask, image


def random_flip(mask, image):
    flip = tf.random.uniform(()) < 0.5
    mask = tf.cond(flip, lambda: tf.gather(tf.image.flip_left_right(mask), FLIP_PERM, axis=-1), lambda: mask)
    image = tf.cond(flip, lambda: tf.image.flip_left_right(image), lambda: image)
    return mask, image


def make_dataset(mask_array, image_array, training=True):
    dataset = tf.data.Dataset.from_tensor_slices((mask_array, image_array))
    if training:
        dataset = dataset.shuffle(
            buffer_size=min(len(mask_array), 10000),
            seed=SEED,
            reshuffle_each_iteration=True
        )
    dataset = dataset.map(to_float, num_parallel_calls=tf.data.AUTOTUNE)
    if training:
        dataset = dataset.map(random_flip, num_parallel_calls=tf.data.AUTOTUNE)
    return dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)


train_ds = make_dataset(mask_train, img_train, training=True)
val_ds = make_dataset(mask_val, img_val, training=False)

In [ ]:
# Comprobación visual del volteo: el color de cada ojo/oreja debe seguir al lado de la persona
example_mask = tf.cast(mask_train[:1], tf.float32) / 255.0
example_img = tf.cast(img_train[:1], tf.float32) / 127.5 - 1.0
flipped_mask = tf.gather(tf.image.flip_left_right(example_mask), FLIP_PERM, axis=-1)
flipped_img = tf.image.flip_left_right(example_img)

fig, axes = plt.subplots(1, 2, figsize=(6, 3.5))
for ax, m, im, title in ((axes[0], example_mask, example_img, "Original"),
                         (axes[1], flipped_mask, flipped_img, "Volteada + canales intercambiados")):
    ax.imshow(im[0, ..., 0], cmap="gray", vmin=-1, vmax=1)
    ax.imshow(mask_to_color(m[0].numpy()), alpha=0.45)
    ax.set_title(title, fontsize=9)
    ax.axis("off")
palette_legend(axes[1])
plt.tight_layout()
plt.show()

---

## **4. Arquitectura**

pix2pix es una **GAN condicional** (CGAN) en la que la condición no es una etiqueta sino una imagen completa: la máscara $x$. El generador $G$ recibe la máscara y produce una cara $G(x)$; el discriminador $D$ recibe **la pareja** (máscara, cara) y decide si la cara es real o generada *para esa máscara*. Así no basta con que la cara sea realista: también tiene que encajar con la máscara.

### **4.1 Generador: de AE 2 a U-Net**

El generador reutiliza los bloques de `AE_complex2`:

- **Encoder**: los mismos 4 bloques (conv `4x4` con stride 2 que reduce + conv `3x3` que refina) con 64-128-256-512 filtros, pero la entrada es la máscara de 8 canales. La resolución baja de `64x64` a `4x4`.
- **Cuello de botella**: se eliminan el `Flatten` y la `Dense` del latente. En AE 2 el vector latente era la única vía de información y por eso tenía que ser un vector muestreable. Aquí la información viaja por la máscara, así que el cuello de botella se queda como un mapa convolucional `4x4x512` que conserva la disposición espacial.
- **Decoder**: los mismos 4 bloques de upsampling bilineal + 2 convoluciones, con **skip connections**: tras cada upsampling se concatena la salida del bloque del encoder de la misma resolución (en el último nivel, la propia máscara de entrada).

En las conclusiones de AE 2 descartamos las skip connections porque al generar desde un $z$ sintético no existe una imagen de entrada de la que copiar. **Aquí la situación es la contraria**: la máscara está disponible tanto al entrenar como al generar, y las skip connections permiten que la posición exacta de ojos, nariz, boca y pelo llegue directamente a las capas que dibujan la cara, sin tener que atravesar el cuello de botella de `4x4`.

**Aleatoriedad.** Una GAN clásica genera a partir de un vector de ruido $z$. Isola et al. observaron que, en pix2pix, el generador aprende a ignorar ese ruido porque la máscara ya determina casi toda la salida. Por eso la aleatoriedad se introduce con **dropout en los primeros bloques del decoder, activo también al generar** (`AlwaysDropout`). Con la misma máscara se obtienen caras algo distintas, aunque la diversidad que se consigue así es limitada (se mide en la sección 10).

In [ ]:
# Bloque convolucional común (el mismo de AE 2)
def conv_block(x, filters, kernel_size=3, strides=1):
    x = layers.Conv2D(filters, kernel_size, strides=strides, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    return layers.LeakyReLU(negative_slope=0.2)(x)


@keras.utils.register_keras_serializable(package=NOTEBOOK_NAME)
class AlwaysDropout(layers.Dropout):
    # Dropout que también actúa al generar: es la fuente de aleatoriedad del generador
    def call(self, inputs, training=None):
        return super().call(inputs, training=True)


def build_generator(name="generator"):
    inputs = keras.Input(shape=(IMG_SIZE, IMG_SIZE, N_CLASSES), name="mask_input")

    # Encoder: 64 -> 32 -> 16 -> 8 -> 4
    x = inputs
    skips = [inputs]
    for filters in (64, 128, 256, 512):
        x = conv_block(x, filters, kernel_size=4, strides=2)
        x = conv_block(x, filters)
        skips.append(x)

    # El último elemento es el cuello de botella (4x4x512); el resto se usan como skips de 8 a 64
    skips = skips[:-1][::-1]

    # Decoder: 4 -> 8 -> 16 -> 32 -> 64
    for level, (filters, skip) in enumerate(zip((512, 256, 128, 64), skips)):
        x = layers.UpSampling2D(size=(2, 2), interpolation="bilinear")(x)
        x = layers.Concatenate()([x, skip])
        x = conv_block(x, filters)
        if level < N_DROPOUT_BLOCKS:
            x = AlwaysDropout(DROPOUT_RATE)(x)
        x = conv_block(x, filters)

    outputs = layers.Conv2D(1, kernel_size=3, padding="same", activation="tanh", name="generated_face")(x)
    return keras.Model(inputs, outputs, name=name)


build_generator().summary()

### **4.2 Discriminador PatchGAN**

Con una pérdida L1 o SSIM la estructura global de la cara se aprende bien, pero los detalles finos salen borrosos, porque el promedio de varias soluciones posibles minimiza el error. pix2pix deja la estructura global a la pérdida de reconstrucción y usa un discriminador especializado en la **textura local**: en vez de una única probabilidad para toda la imagen, el **PatchGAN** devuelve un mapa en el que cada valor juzga si un parche de la imagen es real o falso. La decisión final es la media de todos los parches.

El tamaño del parche es el **campo receptivo** de cada salida, que se calcula capa a capa con $r_l = r_{l-1} + (k_l - 1)\,j_{l-1}$ y $j_l = j_{l-1}\,s_l$ ($k$: kernel, $s$: stride, $j$: salto acumulado entre píxeles):

| Capa | Filtros | Kernel | Stride | Salida | Campo receptivo |
|---|---|---|---|---|---|
| Entrada (máscara + cara) | 9 | | | `64x64` | 1 |
| Conv (sin BN) | 64 | 4 | 2 | `32x32` | 4 |
| Conv + BN | 128 | 4 | 2 | `16x16` | 10 |
| Conv + BN | 256 | 4 | 1 | `16x16` | 22 |
| Conv (logits) | 1 | 4 | 1 | `16x16` | **34** |

El artículo usa parches de `70x70` en imágenes de `256x256` (un 27% del lado). Repetir esa configuración en `64x64` haría que cada salida viese la cara entera; con 2 reducciones cada uno de los `16x16` valores juzga un parche de `34x34` píxeles. La primera capa no lleva BatchNormalization, como en DCGAN y pix2pix.

La salida son *logits* (sin sigmoide): la sigmoide se aplica dentro de la entropía cruzada binaria (`from_logits=True`), que es equivalente a la salida sigmoide de las transparencias pero numéricamente más estable.

In [ ]:
def build_discriminator(name="discriminator"):
    mask_in = keras.Input(shape=(IMG_SIZE, IMG_SIZE, N_CLASSES), name="mask")
    face_in = keras.Input(shape=(IMG_SIZE, IMG_SIZE, 1), name="face")
    x = layers.Concatenate()([mask_in, face_in])

    x = layers.Conv2D(64, 4, strides=2, padding="same")(x)
    x = layers.LeakyReLU(negative_slope=0.2)(x)
    x = conv_block(x, 128, kernel_size=4, strides=2)
    x = conv_block(x, 256, kernel_size=4, strides=1)
    patch_logits = layers.Conv2D(1, 4, strides=1, padding="same", name="patch_logits")(x)
    return keras.Model([mask_in, face_in], patch_logits, name=name)


def receptive_field(model):
    r, j = 1, 1
    for layer in model.layers:
        if isinstance(layer, layers.Conv2D):
            k, s = layer.kernel_size[0], layer.strides[0]
            r, j = r + (k - 1) * j, j * s
    return r


discriminator_check = build_discriminator()
discriminator_check.summary()
print(f"Mapa de salida: {discriminator_check.output_shape[1:3]} | campo receptivo: {receptive_field(discriminator_check)}x{receptive_field(discriminator_check)} píxeles")
del discriminator_check

#### **Hipótesis de diseño**

- **Mismos bloques que AE 2**: el encoder y el decoder ya demostraron en la práctica anterior que reconstruyen bien caras de `64x64`; cambiar solo lo necesario (entrada, cuello de botella y skips) permite atribuir las diferencias a pix2pix y no a otra arquitectura.
- **Skip connections**: la máscara fija dónde va cada rasgo, así que la posición debería conservarse mucho mejor que en un AE sin skips, y las caras deberían respetar el contorno del pelo y la posición de ojos, nariz y boca.
- **Discriminador condicionado**: como $D$ ve la máscara, penaliza las caras realistas que no encajan con ella (por ejemplo, una boca donde la máscara no la tiene).
- **PatchGAN**: la pérdida adversarial debería aportar textura (pelo, contornos) en las zonas que la L1 + SSIM deja borrosas. A cambio, la L1 frente a la cara real puede empeorar, porque la cara generada deja de ser el "promedio" de las posibles.

---

## **5. Protocolo FID / KID**

Se usan las métricas vistas en clase, ambas sobre las características de Inception-v3:

- **FID**: distancia de Fréchet entre las gaussianas que ajustan las características de las caras reales y de las generadas. Mide a la vez realismo y diversidad, pero tiene sesgo con pocas muestras.
- **KID**: discrepancia MMD con un kernel polinómico entre las mismas características. No supone que sean gaussianas y es más fiable con muestras pequeñas, por eso se añade en la evaluación final.

Las caras generadas a partir de las máscaras de **validación** se comparan con las caras reales de validación (selección del mejor generador durante el entrenamiento). Las de **test** se usan solo para las métricas finales.

**Adaptación de Datos e Inferencia**
El cálculo del FID requiere adaptar las imágenes a las especificaciones del modelo de **Inception-v3**, la cual impone restricciones implementadas en `prepare_images_for_fid`:
1. **Rango:** Conversión de intensidades a enteros en $[0, 255]$.
2. **Canales:** Duplicación del canal único en escala de grises a 3 canales RGB.
3. **Formato de tensores:** Transposición del orden de dimensiones de Keras `(B, H, W, C)` al estándar de PyTorch `(B, C, H, W)`.

**Optimización del Cómputo**
El cálculo de FID requiere proyectar tanto las imágenes de referencia como las sintéticas a través de Inception-v3 en cada evaluación. Dado el alto volumen de evaluaciones que vamos a realizar en este notebook, procesar repetidamente las imágenes reales es muy costoso computacionamente y hemos desarrollado gracias a **`Gemini`** un sistema que reduce a la mitad el tiempo de inferencia. La clase `FidEvaluator` implementa un sistema de almacenamiento en caché mediante el parámetro `reset_real_features=False` de Torchmetrics:

* **Extracción persistente:** Proyecta la referencia real una vez por cada tamaño $N$ y retiene sus estadísticas ($\mu_r, \Sigma_r$) en memoria.
* **Reinicio:** En cada llamada de evaluación, `fid.reset()` elimina solo las estadísticas de las imágenes sintéticas por lo que solo necesitamos la inferencia sobre las nuevas muestras generadas.
* **Consistencia:** Se ejecuta un test comparando la salida de `FidEvaluator` frente a la implementación sin estado `calculate_fid`. La caché solo se activa (`USE_FID_CACHE=True`) si la diferencia es inferior a $0.1$ para estar seguros de que funciona.

**Suelo de referencia.** Como en AE 2 se calculaba el "techo del decoder", aquí se calcula el FID entre caras **reales** de entrenamiento y caras reales de test. Es el valor al que podría aspirar un generador perfecto con este tamaño de muestra.

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"


def to_uint8(images_float):
    # [-1, 1] -> [0, 255]
    images_float = np.asarray(images_float)
    return np.clip(np.round((images_float + 1.0) * 127.5), 0, 255).astype(np.uint8)


def prepare_images_for_fid(images):
    images = np.asarray(images)
    if images.dtype != np.uint8:
        images = to_uint8(images)
    if images.ndim == 3:
        images = images[..., np.newaxis]
    images = np.repeat(images, 3, axis=-1)
    images = np.transpose(images, (0, 3, 1, 2))
    return torch.from_numpy(np.ascontiguousarray(images))


def calculate_fid(generated_images, real_images, batch_size=FID_BATCH_SIZE):
    fid = FrechetInceptionDistance(feature=2048, normalize=False).to(device)
    real_images = prepare_images_for_fid(real_images)
    generated_images = prepare_images_for_fid(generated_images)
    with torch.inference_mode():
        for start in range(0, len(real_images), batch_size):
            fid.update(real_images[start:start + batch_size].to(device), real=True)
        for start in range(0, len(generated_images), batch_size):
            fid.update(generated_images[start:start + batch_size].to(device), real=False)
        return fid.compute().item()


class FidEvaluator:
    def __init__(self, real_images, batch_size=FID_BATCH_SIZE):
        self.real_images = real_images
        self.batch_size = batch_size
        self.metrics = {}

    def _update_real(self, metric, n):
        real = prepare_images_for_fid(self.real_images[:n])
        with torch.inference_mode():
            for start in range(0, len(real), self.batch_size):
                metric.update(real[start:start + self.batch_size].to(device), real=True)

    def _metric_for(self, kind, n):
        key = (kind, n)
        if key not in self.metrics:
            if kind == "fid":
                metric = FrechetInceptionDistance(feature=2048, normalize=False, reset_real_features=False)
            else:
                metric = KernelInceptionDistance(feature=2048, subset_size=min(KID_SUBSET_SIZE, n),
                                                 normalize=False, reset_real_features=False)
            metric = metric.to(device)
            self._update_real(metric, n)
            self.metrics[key] = metric
        return self.metrics[key]

    def _compute(self, kind, generated_images):
        metric = self._metric_for(kind, len(generated_images))
        metric.reset()  # mantenemos las características reales y eliminamos las generadas
        generated = prepare_images_for_fid(generated_images)
        with torch.inference_mode():
            for start in range(0, len(generated), self.batch_size):
                metric.update(generated[start:start + self.batch_size].to(device), real=False)
            return metric.compute()

    def compute(self, generated_images):
        return self._compute("fid", generated_images).item()

    def compute_kid(self, generated_images):
        torch.manual_seed(SEED)   # KID promedia subconjuntos aleatorios
        kid_mean, kid_std = self._compute("kid", generated_images)
        return kid_mean.item(), kid_std.item()


fid_val = FidEvaluator(img_val)
fid_test = FidEvaluator(img_test)

# Validación de la caché frente a calculate_fid (y repitiendo para comprobar el reset)
check_imgs = img_train[:len(img_val)]
fid_plain = calculate_fid(check_imgs, img_val)
fid_cached_1 = fid_val.compute(check_imgs)
fid_cached_2 = fid_val.compute(check_imgs)
USE_FID_CACHE = abs(fid_cached_1 - fid_plain) < 0.1 and abs(fid_cached_2 - fid_plain) < 0.1
print(f"calculate_fid: {fid_plain:.4f} | caché: {fid_cached_1:.4f} y {fid_cached_2:.4f} | usar caché: {USE_FID_CACHE}")


def fid_images(generated_images, evaluator):
    n = len(generated_images)
    assert n <= len(evaluator.real_images), "No hay suficientes imágenes reales como referencia."
    if USE_FID_CACHE:
        return evaluator.compute(generated_images)
    return calculate_fid(generated_images, evaluator.real_images[:n])

In [ ]:
# Suelo: caras reales de entrenamiento frente a caras reales de test
floor_imgs = img_train[:len(img_test)]
fid_floor = fid_images(floor_imgs, fid_test)
kid_floor = fid_test.compute_kid(floor_imgs)
print(f"Suelo real vs real (N={len(img_test)}): FID = {fid_floor:.2f} | KID = {kid_floor[0]:.4f} ± {kid_floor[1]:.4f}")

---

## **6. Pérdidas y entrenamiento**

Con $x$ la máscara e $y$ la cara real, el objetivo de pix2pix es

$$\mathcal{L}_{cGAN}(G, D) = \mathbb{E}_{x,y}\left[\log D(x, y)\right] + \mathbb{E}_{x}\left[\log\left(1 - D(x, G(x))\right)\right]$$

$$G^* = \arg\min_G \max_D \; \mathcal{L}_{cGAN}(G, D) + \lambda\,\mathcal{L}_{rec}(G), \qquad \mathcal{L}_{rec} = 0.5\,\lVert y - G(x)\rVert_1 + 0.5\,\left(1 - \text{SSIM}(y, G(x))\right)$$

Como pérdida de reconstrucción se mantiene la L1 + SSIM de AE 2 (calculada en `[0, 1]`) en lugar de la L1 sola del artículo; la SSIM ayuda a conservar la estructura local de la cara.

Cómo se implementa:

- **Discriminador**: minimiza $\tfrac{1}{2}\left[\text{BCE}(1, D(x, y)) + \text{BCE}(0, D(x, G(x)))\right]$, aplicada a cada parche. El factor $\tfrac{1}{2}$ frena a $D$ respecto a $G$, como en el artículo.
- **Generador**: minimiza $\text{BCE}(1, D(x, G(x))) + \lambda\,\mathcal{L}_{rec}$. Es la versión **no saturante**: en vez de minimizar $\log(1 - D(x, G(x)))$ se maximiza $\log D(x, G(x))$. Al principio $D$ rechaza las caras generadas con facilidad, $\log(1 - D)$ está casi plano y su gradiente se desvanece; la versión no saturante da gradientes fuertes justo en ese momento.
- **Optimizadores**: Adam con `lr = 2e-4` y `beta_1 = 0.5` para las dos redes, y un paso de $D$ por cada paso de $G$.
- **Seguimiento**: además de las pérdidas se registran $D(x, y)$ y $D(x, G(x))$ como probabilidades medias. En el equilibrio de Nash el discriminador óptimo es $D^*(x) = \frac{p_{data}}{p_{data} + p_g} = 0.5$; si $D(x, y) \to 1$ y $D(x, G(x)) \to 0$ el discriminador está ganando, y si las pérdidas se vuelven `NaN` o constantes hay explosión o desvanecimiento del gradiente.
- **Sin `EarlyStopping` ni `ReduceLROnPlateau`**: en una GAN las pérdidas no bajan de forma monótona y no indican la calidad. En su lugar, un callback calcula el FID en validación cada `FID_EVERY` épocas y guarda el mejor generador.

In [ ]:
def ssim_l1_loss(y_true, y_pred):
    # Caras en [-1, 1] -> [0, 1], igual que en AE 2
    y_true = (y_true + 1.0) / 2.0
    y_pred = (y_pred + 1.0) / 2.0
    l1 = tf.reduce_mean(tf.abs(y_true - y_pred))
    ssim = tf.reduce_mean(tf.image.ssim(y_true, y_pred, max_val=1.0))
    return 0.5 * l1 + 0.5 * (1.0 - ssim), l1, ssim


class Pix2Pix(keras.Model):
    def __init__(self, generator, discriminator, lambda_rec=LAMBDA_REC, **kwargs):
        super().__init__(**kwargs)
        self.generator = generator
        self.discriminator = discriminator
        self.lambda_rec = lambda_rec
        self.bce = keras.losses.BinaryCrossentropy(from_logits=True)

        self.rec_tracker = keras.metrics.Mean(name="rec_loss")
        self.l1_tracker = keras.metrics.Mean(name="l1")
        self.ssim_tracker = keras.metrics.Mean(name="ssim")
        self.g_adv_tracker = keras.metrics.Mean(name="g_adv")
        self.d_loss_tracker = keras.metrics.Mean(name="d_loss")
        self.d_real_tracker = keras.metrics.Mean(name="d_real")
        self.d_fake_tracker = keras.metrics.Mean(name="d_fake")

    def compile(self, g_optimizer, d_optimizer):
        super().compile()
        self.g_optimizer = g_optimizer
        self.d_optimizer = d_optimizer

    @property
    def metrics(self):
        return [self.rec_tracker, self.l1_tracker, self.ssim_tracker,
                self.g_adv_tracker, self.d_loss_tracker, self.d_real_tracker, self.d_fake_tracker]

    def train_step(self, data):
        mask, real = data
        with tf.GradientTape() as g_tape, tf.GradientTape() as d_tape:
            fake = self.generator(mask, training=True)
            d_real = self.discriminator([mask, real], training=True)
            d_fake = self.discriminator([mask, fake], training=True)

            rec, l1, ssim = ssim_l1_loss(real, fake)
            g_adv = self.bce(tf.ones_like(d_fake), d_fake)
            g_loss = g_adv + self.lambda_rec * rec
            d_loss = 0.5 * (self.bce(tf.ones_like(d_real), d_real) +
                            self.bce(tf.zeros_like(d_fake), d_fake))

        g_grads = g_tape.gradient(g_loss, self.generator.trainable_variables)
        d_grads = d_tape.gradient(d_loss, self.discriminator.trainable_variables)
        self.g_optimizer.apply_gradients(zip(g_grads, self.generator.trainable_variables))
        self.d_optimizer.apply_gradients(zip(d_grads, self.discriminator.trainable_variables))

        self.rec_tracker.update_state(rec)
        self.l1_tracker.update_state(l1)
        self.ssim_tracker.update_state(ssim)
        self.g_adv_tracker.update_state(g_adv)
        self.d_loss_tracker.update_state(d_loss)
        self.d_real_tracker.update_state(tf.reduce_mean(tf.sigmoid(d_real)))
        self.d_fake_tracker.update_state(tf.reduce_mean(tf.sigmoid(d_fake)))
        return {m.name: m.result() for m in self.metrics}

    def test_step(self, data):
        mask, real = data
        fake = self.generator(mask, training=False)
        rec, l1, ssim = ssim_l1_loss(real, fake)
        self.rec_tracker.update_state(rec)
        self.l1_tracker.update_state(l1)
        self.ssim_tracker.update_state(ssim)
        return {"rec_loss": self.rec_tracker.result(),
                "l1": self.l1_tracker.result(),
                "ssim": self.ssim_tracker.result()}

In [ ]:
def generate(generator, mask_array, seed=None):
    # Máscaras uint8 -> caras uint8 (N, 64, 64). El dropout hace que cada llamada sea distinta
    if seed is not None:
        keras.utils.set_random_seed(seed)
    masks_float = mask_array.astype(np.float32) / 255.0
    out = generator.predict(masks_float, batch_size=256, verbose=0)
    return to_uint8(out[..., 0])


class FidCheckpoint(keras.callbacks.Callback):
    # Calcula el FID en validación cada `every` épocas y guarda el generador con el mejor FID
    def __init__(self, generator, best_path, every=FID_EVERY, total_epochs=EPOCHS):
        super().__init__()
        self.generator = generator
        self.best_path = best_path
        self.every = every
        self.total_epochs = total_epochs
        self.history = []
        self.best_fid = np.inf
        self.best_epoch = None

    def on_epoch_end(self, epoch, logs=None):
        if (epoch + 1) % self.every and epoch + 1 != self.total_epochs:
            return
        fid = fid_images(generate(self.generator, mask_val, seed=SEED), fid_val)
        self.history.append({"epoch": epoch + 1, "fid": fid})
        improved = fid < self.best_fid
        if improved:
            self.best_fid, self.best_epoch = fid, epoch + 1
            self.generator.save(self.best_path)
        print(f" - FID val: {fid:.2f}" + (" (mejor, guardado)" if improved else f" (mejor: {self.best_fid:.2f} en época {self.best_epoch})"))

In [ ]:
keras.utils.set_random_seed(SEED)
generator = build_generator()
discriminator = build_discriminator()

pix2pix = Pix2Pix(generator, discriminator, name="pix2pix")
pix2pix.compile(
    g_optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE, beta_1=BETA_1),
    d_optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE, beta_1=BETA_1)
)
fid_callback = FidCheckpoint(generator, GENERATOR_BEST_PATH)

print(f"Generador:     {generator.count_params():,} parámetros")
print(f"Discriminador: {discriminator.count_params():,} parámetros")

In [ ]:
if TRAIN_MODEL:
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    t0 = time.time()
    history_p2p = pix2pix.fit(
        train_ds,
        epochs=EPOCHS,
        validation_data=val_ds,
        callbacks=[fid_callback]
    )
    train_time = time.time() - t0
    print(f"Entrenamiento: {train_time / 60:.1f} min | mejor FID val "
          f"{fid_callback.best_fid:.2f} en la época {fid_callback.best_epoch}")
else:
    print("TRAIN_MODEL = False: no se entrena; los modelos se cargan en la celda de carga.")

In [ ]:
if TRAIN_MODEL:
    # El mejor generador ya lo ha guardado el callback; aquí se guarda el estado final
    generator.save(GENERATOR_LAST_PATH)
    discriminator.save(DISCRIMINATOR_LAST_PATH)
    print(f"Modelos guardados en {CHECKPOINT_DIR}/")

In [ ]:
if TRAIN_MODEL:
    record = {
        "history": {k: [float(v) for v in values] for k, values in history_p2p.history.items()},
        "fid_val": fid_callback.history,
        "best_epoch": fid_callback.best_epoch,
        "best_fid_val": float(fid_callback.best_fid),
        "train_time_s": train_time,
        "n_params_generator": int(generator.count_params()),
        "n_params_discriminator": int(discriminator.count_params()),
    }
    with open(HISTORY_PATH, "w") as f:
        json.dump(record, f, indent=2)
    print(f"Historial guardado en {HISTORY_PATH}")

In [ ]:
# Para evaluar se usa el mejor generador (por FID en validación), no el de la última época
required = [GENERATOR_BEST_PATH, HISTORY_PATH]
missing = [p for p in required if not os.path.exists(p)]
if missing:
    raise FileNotFoundError(
        f"Faltan ficheros guardados: {missing}. "
        "Ejecuta una vez con TRAIN_MODEL=True para entrenar y guardarlos."
    )

# compile=False: la pérdida no hace falta para generar
best_generator = keras.models.load_model(
    GENERATOR_BEST_PATH, compile=False, custom_objects={"AlwaysDropout": AlwaysDropout}
)
with open(HISTORY_PATH) as f:
    record = json.load(f)

assert best_generator.input_shape[-1] == N_CLASSES, \
    "El generador guardado no tiene un canal por clase. Reentrena con TRAIN_MODEL=True."

print("Modelo cargado:")
print(f"  generador: {GENERATOR_BEST_PATH}")
print(f"  historial: {HISTORY_PATH} ({len(record['history']['l1'])} épocas)")
print(f"  mejor época {record['best_epoch']} | FID val {record['best_fid_val']:.2f} | "
      f"{record['train_time_s'] / 60:.1f} min de entrenamiento")

---

## **7. Curvas de entrenamiento**

In [ ]:
h = record["history"]
epochs = np.arange(1, len(h["l1"]) + 1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))
axes[0].plot(epochs, h["l1"], label="Train")
axes[0].plot(epochs, h["val_l1"], "--", label="Val")
axes[1].plot(epochs, h["ssim"], label="Train")
axes[1].plot(epochs, h["val_ssim"], "--", label="Val")
fid_hist = record["fid_val"]
axes[2].plot([p["epoch"] for p in fid_hist], [p["fid"] for p in fid_hist], "o-", label="FID val")
axes[2].axvline(record["best_epoch"], color="gray", linestyle=":", label=f"mejor época ({record['best_epoch']})")

axes[0].set_title("L1 (en [0, 1])")
axes[1].set_title("SSIM")
axes[2].set_title("FID en validación")
for ax in axes:
    ax.set_xlabel("Época")
    ax.legend()
plt.tight_layout()
save_fig("curvas_reconstruccion_fid")
plt.show()

In [ ]:
# Dinámica adversarial de pix2pix
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(epochs, h["d_loss"], label="D: pérdida")
axes[0].plot(epochs, h["g_adv"], label="G: pérdida adversarial")
axes[0].plot(epochs, np.array(h["rec_loss"]) * LAMBDA_REC / 10, label="G: λ·L_rec / 10")
axes[0].set_title("Pérdidas de pix2pix")
axes[0].legend()

axes[1].plot(epochs, h["d_real"], label="D(x, y): caras reales")
axes[1].plot(epochs, h["d_fake"], label="D(x, G(x)): caras generadas")
axes[1].axhline(0.5, color="gray", linestyle="--", label="equilibrio (0.5)")
axes[1].set_ylim(0, 1)
axes[1].set_title("Salida media del discriminador")
axes[1].legend()
for ax in axes:
    ax.set_xlabel("Época")
plt.tight_layout()
save_fig("curvas_adversariales")
plt.show()

---

## **8. Resultados visuales en test**

Máscaras de test (nunca vistas en entrenamiento ni usadas para elegir el modelo), las caras generadas por pix2pix y la cara real correspondiente.

In [ ]:
n_show = 8
show_idx = np.random.default_rng(SEED).choice(len(mask_test), size=n_show, replace=False)
generated_show = generate(best_generator, mask_test[show_idx], seed=SEED)

fig, axes = plt.subplots(3, n_show, figsize=(1.7 * n_show, 1.9 * 3))
for c, idx in enumerate(show_idx):
    axes[0, c].imshow(mask_to_color(mask_test[idx]))
    axes[1, c].imshow(generated_show[c], cmap="gray", vmin=0, vmax=255)
    axes[2, c].imshow(img_test[idx, ..., 0], cmap="gray", vmin=0, vmax=255)
for r, title in enumerate(("Máscara", "pix2pix", "Real")):
    axes[r, 0].set_title(title, loc="left", fontsize=10)
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
save_fig("resultados_test")
plt.show()

---

## **9. Métricas objetivas en test**

Se generan las caras de las `N` máscaras de test `N_EVAL_REPEATS` veces (semillas distintas) y se calcula:

- **FID y KID** frente a las caras reales de test: realismo y diversidad del conjunto.
- **L1, SSIM y PSNR** frente a la cara real de **cada** máscara: fidelidad a la imagen original. No son el objetivo principal (una cara puede ser realista y coherente con la máscara y aun así distinta de la real), pero muestran el compromiso entre fidelidad y nitidez que introduce la pérdida adversarial.

In [ ]:
def paired_metrics(generated_u8, real_u8):
    g = tf.convert_to_tensor(generated_u8[..., None], tf.float32) / 255.0
    r = tf.convert_to_tensor(real_u8, tf.float32) / 255.0
    return {
        "l1": float(tf.reduce_mean(tf.abs(g - r))),
        "ssim": float(tf.reduce_mean(tf.image.ssim(r, g, max_val=1.0))),
        "psnr": float(tf.reduce_mean(tf.image.psnr(r, g, max_val=1.0))),
    }


# Batería: N_EVAL_REPEATS generaciones del conjunto de test
t0 = time.time()
runs = []
for rep in range(N_EVAL_REPEATS):
    gen = generate(best_generator, mask_test, seed=SEED + rep)
    kid_mean, _ = fid_test.compute_kid(gen)
    runs.append({"fid": fid_images(gen, fid_test), "kid": kid_mean, **paired_metrics(gen, img_test)})
df_runs = pd.DataFrame(runs)

test_results = {}
for metric in df_runs.columns:
    test_results[f"{metric}_mean"] = float(df_runs[metric].mean())
    test_results[f"{metric}_std"] = float(df_runs[metric].std(ddof=1))

df_test = pd.DataFrame({
    "media": df_runs.mean(),
    "desviación": df_runs.std(ddof=1),
})
print(f"Métricas en test ({N_EVAL_REPEATS} repeticiones, {time.time() - t0:.0f}s):")
print(df_test.round(4).to_string())
print(f"\nSuelo real vs real (N={len(img_test)}): FID = {fid_floor:.2f} | KID = {kid_floor[0]:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 2.4))
for ax, metric, floor in ((axes[0], "fid", fid_floor), (axes[1], "kid", kid_floor[0])):
    ax.barh(["pix2pix"], [test_results[f"{metric}_mean"]], xerr=[test_results[f"{metric}_std"]],
            capsize=3, color="tab:green")
    ax.axvline(floor, color="r", linestyle="--", label=f"suelo real vs real: {floor:.3g}")
    ax.set_title(f"{metric.upper()} en test (menor es mejor)")
    ax.legend(fontsize=8)
plt.tight_layout()
save_fig("metricas_test")
plt.show()

---

## **10. Diversidad: varias caras para la misma máscara**

Como la aleatoriedad viene del dropout, generar varias veces con la misma máscara da caras distintas. Se muestra para algunas máscaras de test junto con el **mapa de desviación típica por píxel** entre muestras (las zonas claras son las que más cambian), y se resume con la desviación típica media sobre `N_DIVERSITY_MASKS` máscaras.

In [ ]:
def diversity_samples(generator, mask_array, n_samples, seed=SEED):
    repeated = np.repeat(mask_array, n_samples, axis=0)
    gen = generate(generator, repeated, seed=seed)
    return gen.reshape(len(mask_array), n_samples, IMG_SIZE, IMG_SIZE)


samples = diversity_samples(best_generator, mask_test[:N_DIVERSITY_MASKS], N_DIVERSITY).astype(np.float32) / 255.0
diversity = float(samples.std(axis=1).mean())
print(f"Desviación típica media por píxel entre {N_DIVERSITY} muestras = {diversity:.4f}")

n_masks_show = 3
samples = diversity_samples(best_generator, mask_test[show_idx[:n_masks_show]], N_DIVERSITY)
fig, axes = plt.subplots(n_masks_show, N_DIVERSITY + 2, figsize=(1.6 * (N_DIVERSITY + 2), 1.8 * n_masks_show))
for r in range(n_masks_show):
    axes[r, 0].imshow(mask_to_color(mask_test[show_idx[r]]))
    for c in range(N_DIVERSITY):
        axes[r, c + 1].imshow(samples[r, c], cmap="gray", vmin=0, vmax=255)
    axes[r, -1].imshow(samples[r].astype(np.float32).std(axis=0), cmap="magma")
    for ax in axes[r]:
        ax.axis("off")
axes[0, 0].set_title("Máscara", fontsize=9)
axes[0, 1].set_title(f"pix2pix: {N_DIVERSITY} muestras →", loc="left", fontsize=9)
axes[0, -1].set_title("Desv. por píxel", fontsize=9)
plt.tight_layout()
save_fig("diversidad")
plt.show()

---

## **11. Originalidad: ¿copia caras del entrenamiento?**

Un buen generador no debe limitarse a devolver caras del conjunto de entrenamiento. Para cada cara generada a partir de una máscara de test se busca la **cara de entrenamiento más cercana** (distancia euclídea en píxeles, expresada como RMS en `[0, 1]`) y se compara con la misma distancia calculada para las caras **reales** de test. Si el generador copiase, sus caras estarían mucho más cerca del entrenamiento que las caras reales nuevas.

In [ ]:
train_flat = img_train.reshape(len(img_train), -1).astype(np.float32) / 255.0
train_sq = (train_flat ** 2).sum(axis=1)


def nearest_train(query_u8, batch=250):
    q_all = query_u8.reshape(len(query_u8), -1).astype(np.float32) / 255.0
    idx, dist = [], []
    for start in range(0, len(q_all), batch):
        q = q_all[start:start + batch]
        d2 = (q ** 2).sum(axis=1)[:, None] + train_sq[None, :] - 2.0 * q @ train_flat.T
        best = d2.argmin(axis=1)
        idx.append(best)
        dist.append(np.sqrt(np.maximum(d2[np.arange(len(q)), best], 0.0) / q.shape[1]))
    return np.concatenate(idx), np.concatenate(dist)


gen_orig = generate(best_generator, mask_test[:N_ORIGINALITY], seed=SEED)
nn_idx_gen, nn_dist_gen = nearest_train(gen_orig)
nn_idx_real, nn_dist_real = nearest_train(img_test[:N_ORIGINALITY, ..., 0])
print(f"Distancia RMS a la cara de entrenamiento más cercana (N={N_ORIGINALITY}):")
print(f"  generadas pix2pix: media {nn_dist_gen.mean():.4f} | mínimo {nn_dist_gen.min():.4f}")
print(f"  reales de test:    media {nn_dist_real.mean():.4f} | mínimo {nn_dist_real.min():.4f}")

fig = plt.figure(figsize=(14, 4.2))
ax_hist = fig.add_subplot(1, 2, 1)
ax_hist.hist(nn_dist_real, bins=40, alpha=0.6, label="reales de test")
ax_hist.hist(nn_dist_gen, bins=40, alpha=0.6, label="generadas pix2pix")
ax_hist.set_xlabel("Distancia RMS a la cara de entrenamiento más cercana")
ax_hist.set_title("Originalidad")
ax_hist.legend()

n_pairs = 5
for c in range(n_pairs):
    for r, (img, title) in enumerate(((gen_orig[c], "Generada"),
                                      (img_train[nn_idx_gen[c], ..., 0], "Más cercana (train)"))):
        ax = fig.add_subplot(2, 2 * n_pairs, (r * 2 * n_pairs) + n_pairs + c + 1)
        ax.imshow(img, cmap="gray", vmin=0, vmax=255)
        ax.axis("off")
        if c == 0:
            ax.set_title(title, loc="left", fontsize=8)
plt.tight_layout()
save_fig("originalidad")
plt.show()

---

## **12. Edición de máscaras**

Como la entrada es una máscara, se puede **editar a mano** y ver cómo responde el generador. Se parte de una máscara de test y se modifican sus canales: quitar el pelo, agrandar la boca, añadir un gorro tomado de otra máscara o cambiar el pelo por el de otra persona. Todas las caras se generan con la misma semilla para que las diferencias se deban solo a la máscara.

In [ ]:
hair_c, hat_c, mouth_c = CLASSES.index("hair"), CLASSES.index("hat"), CLASSES.index("mouth")
present_test = mask_test.max(axis=(1, 2)) > 127
hair_area_test = (mask_test[..., hair_c] > 127).mean(axis=(1, 2))

base_i = int(np.where(present_test[:, hair_c] & present_test[:, mouth_c] & ~present_test[:, hat_c]
                      & (hair_area_test > 0.15) & (hair_area_test < 0.35))[0][0])
hair_donor_i = int(np.argmax(hair_area_test))                              # pelo más largo del test
hat_donor_i = int(np.where(mask_train.max(axis=(1, 2))[:, hat_c] > 127)[0][0])

base = mask_test[base_i]
edits = {"Original": base.copy()}

m = base.copy(); m[..., hair_c] = 0
edits["Sin pelo"] = m

m = base.copy(); m[..., mouth_c] = cv2.dilate(base[..., mouth_c], np.ones((3, 3), np.uint8), iterations=2)
edits["Boca más grande"] = m

m = base.copy(); m[..., hat_c] = mask_train[hat_donor_i, ..., hat_c]
m[..., hair_c] = np.where(m[..., hat_c] > 127, 0, m[..., hair_c])
edits["Con gorro"] = m

m = base.copy(); m[..., hair_c] = mask_test[hair_donor_i, ..., hair_c]
edits["Pelo de otra persona"] = m

edited = np.stack(list(edits.values()))
edited_faces = np.concatenate([generate(best_generator, edited[i:i + 1], seed=SEED) for i in range(len(edited))])

fig, axes = plt.subplots(2, len(edits), figsize=(2.3 * len(edits), 4.8))
for c, (title, m) in enumerate(edits.items()):
    axes[0, c].imshow(mask_to_color(m))
    axes[0, c].set_title(title, fontsize=9)
    axes[1, c].imshow(edited_faces[c], cmap="gray", vmin=0, vmax=255)
for ax in axes.flat:
    ax.axis("off")
plt.tight_layout()
save_fig("edicion_mascaras")
plt.show()

---

## **13. Métricas guardadas**

In [ ]:
metrics = {
    "split": {"train": len(img_train), "val": len(img_val), "test": len(img_test)},
    "config": {"epochs": EPOCHS, "batch_size": BATCH_SIZE, "learning_rate": LEARNING_RATE, "beta_1": BETA_1,
               "lambda_rec": LAMBDA_REC, "dropout_rate": DROPOUT_RATE, "n_dropout_blocks": N_DROPOUT_BLOCKS},
    "floor_real_vs_real": {"fid": float(fid_floor), "kid": float(kid_floor[0])},
    "best_epoch": record["best_epoch"],
    "best_fid_val": record["best_fid_val"],
    "train_time_min": record["train_time_s"] / 60,
    "n_params_generator": record["n_params_generator"],
    "n_params_discriminator": record["n_params_discriminator"],
    "test": test_results,
    "diversity_pixel_std": diversity,
    "originality_rms_nearest_train": {"generated_mean": float(nn_dist_gen.mean()),
                                      "real_test_mean": float(nn_dist_real.mean())},
}
os.makedirs(RESULTS_DIR, exist_ok=True)
with open(METRICS_PATH, "w") as f:
    json.dump(metrics, f, indent=2)
print(f"Métricas guardadas en {METRICS_PATH}")
print(json.dumps(metrics, indent=2))

---

## **14. Conclusiones**

*(Pendiente de completar con los resultados del entrenamiento.)*

### **Referencias**

- P. Isola, J.-Y. Zhu, T. Zhou y A. A. Efros. *Image-to-Image Translation with Conditional Adversarial Networks*. CVPR, 2017.
- O. Ronneberger, P. Fischer y T. Brox. *U-Net: Convolutional Networks for Biomedical Image Segmentation*. MICCAI, 2015.
- I. Goodfellow et al. *Generative Adversarial Nets*. NeurIPS, 2014.
- M. Mirza y S. Osindero. *Conditional Generative Adversarial Nets*. arXiv:1411.1784, 2014.
- A. Radford, L. Metz y S. Chintala. *Unsupervised Representation Learning with Deep Convolutional GANs (DCGAN)*. ICLR, 2016.
- M. Heusel et al. *GANs Trained by a Two Time-Scale Update Rule Converge to a Local Nash Equilibrium (FID)*. NeurIPS, 2017.
- M. Bińkowski et al. *Demystifying MMD GANs (KID)*. ICLR, 2018.
- C.-H. Lee, Z. Liu, L. Wu y P. Luo. *MaskGAN: Towards Diverse and Interactive Facial Image Manipulation (CelebAMask-HQ)*. CVPR, 2020.